In [6]:
!pip install scipy seaborn tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 33.2 MB/s  0:00:00.3 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [seaborn]


In [7]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
import seaborn as sns
from itertools import product
from tqdm import tqdm, trange

In [8]:
%load_ext autoreload
%autoreload 2

In [9]:
from sim import simulate_errors

ModuleNotFoundError: No module named 'sim'

In [ ]:
rng = np.random.default_rng(42)

eps = 0.001
phi_max = 0.1

## Finding only N first

In [ ]:
def find_n_binary_search(phi, phi_max, m_exploration=400, safeguard=1, conf=0.95):
    N_min = np.pi//(2*phi_max)
    N_max = np.pi//(2*phi_max/10)
    N = N_min
    
    phi_hat = simulate_errors(rng, phi, m_exploration, N)
    budget_used = m_exploration * N
    
    phi_1 = norm.ppf(1 - conf, phi_hat, np.sqrt(1 / (4 * m_exploration * N**2)))
    
    lb = N_min
    ub = N_max
    N += (ub - N) // 2
    
    done = False
    while not done:
        phi_hat = simulate_errors(rng, phi, m_exploration, N)
        budget_used += m_exploration * N
        
        temp_N = N
        if phi_hat < phi_1:
            N -= (N - lb) // 2
            ub = temp_N
        else:
            N += (ub - N) // 2
            lb = temp_N
            
            # update phi_1
            phi_1 = norm.ppf(1 - conf, phi_hat, np.sqrt(1 / (4 * m_exploration * N**2)))
        
        if temp_N == N or N < N_min or N > N_max:
            done = True
    
    N -= safeguard # to avoid overshooting
    
    return N, budget_used

In [2]:
R = 1000

params = {
    "m_exploration": 10_000,
    "safeguard": 1,
    "conf": 0.95
}

results = []
for _ in range(R):
    phi = rng.uniform(phi_max/10, phi_max)
    N_opt = np.pi//(2*phi)

    N, budget_used = find_n_binary_search(phi, phi_max, **params)

    # print(f"{phi = :.3f} {N_opt = }, {N = }")
    result = {
        "N_opt": N_opt,
        "N": N,
        "err": N_opt - N,
        "budget": budget_used
    }
    results.append(result)

NameError: name 'rng' is not defined

In [3]:
errs = np.array([r["err"] for r in results])
budgets = np.array([r["budget"] for r in results])

plt.figure(figsize=(16, 8))

# symmetric bins around 0
# max_abs = max(abs(errs.min()), abs(errs.max()))
bins = np.arange(-errs.min() - 0.5, errs.max() + 1.5, 1)

plt.hist(errs, bins=bins, edgecolor="black", alpha=0.7)

# visual markers
plt.axvline(0, color="black", linestyle="--", linewidth=1)
plt.axvspan(bins[0], 0, color="red", alpha=0.15, label="Overshoot (N > N_opt)")
plt.axvspan(0, bins[-1], color="green", alpha=0.15, label="Undershoot (N < N_opt)")

plt.xlabel(r"$N_{\mathrm{opt}} - N$")
plt.ylabel("Frequency")
plt.title(
    rf"Binary search estimator error distribution ($R={R}$)"
)

plt.legend()
plt.tight_layout()
plt.show()

overshoot_rate = np.mean(errs < 0)
perfect_rate = np.mean(errs == 0)
mean_undershoot = np.mean(errs[errs > 0]) if np.any(errs > 0) else 0
mean_budget = np.mean(budgets)

print(f"Overshoot rate: {overshoot_rate:.2%}")
print(f"Exact hit rate: {perfect_rate:.2%}")
print(f"Mean undershoot: {mean_undershoot:.2}")
print(f"Mean budget: {mean_budget}")



ValueError: zero-size array to reduction operation minimum which has no identity

<Figure size 1600x800 with 0 Axes>

## Parameter grid search

In [8]:
m_vals = [200, 400, 500, 1000, 2000]
conf_vals = [0.9, 0.95, 0.975]
safeguard_vals = [0, 1, 2, 3]

grid_results = []

for m in m_vals:
    for conf in conf_vals:
        for sg in safeguard_vals:
            errs = []
            budgets = []

            for _ in range(300):  # fewer trials for speed
                phi = rng.uniform(phi_max / 10, phi_max)
                N_opt = np.pi // (2 * phi)

                N, budget = find_n_binary_search(
                    phi,
                    phi_max,
                    m_exploration=m,
                    conf=conf,
                    safeguard=sg,
                )

                errs.append(N_opt - N)
                budgets.append(budget)

            errs = np.array(errs)

            overshoot_rate = np.mean(errs < 0)
            mean_budget = np.mean(budgets)
            mean_undershoot = np.mean(errs[errs > 0]) if np.any(errs > 0) else 0

            # loss = 1000 * overshoot_rate + mean_budget

            grid_results.append({
                "m": m,
                "conf": conf,
                "safeguard": sg,
                "overshoot_rate": overshoot_rate,
                "mean_budget": mean_budget,
                "mean_undershoot": mean_undershoot,
                # "loss": loss,
            })

# best = min(grid_results, key=lambda x: x["loss"])
# best


In [9]:
# 1. Minimize overshoot rate
# min_overshoot = min(r["overshoot_rate"] for r in grid_results)

# candidates = [
    # r for r in grid_results
    # if np.isclose(r["overshoot_rate"], min_overshoot)
# ]

candidates = [
    r for r in grid_results
    if r["overshoot_rate"] <= 0.02
]


# 2. Among those, minimize budget
min_budget = min(r["mean_budget"] for r in candidates)

candidates = [
    r for r in candidates
    if np.isclose(r["mean_budget"], min_budget)
]

# 3. Among those, minimize undershoot
best = min(candidates, key=lambda r: r["mean_undershoot"])

best


{'m': 400,
 'conf': 0.9,
 'safeguard': 3,
 'overshoot_rate': 0.013333333333333334,
 'mean_budget': 144304.0,
 'mean_undershoot': 4.534722222222222}

## Find estimate $\hat\phi$

In [10]:
from binary_search import find_phi_binary_search

In [11]:
# def grid_search_algorithm(
#     find_phi_fn,
#     param_grid,
#     *,
#     R=300,
#     eps=1e-3,
#     phi_min=None,
#     phi_max=0.1,
#     success_threshold=0.95,
#     lambda_budget=1e-7,
#     rng=None,
#     verbose=True,
# ):
#     """
#     Generic grid search for phase-estimation algorithms.

#     Parameters
#     ----------
#     find_phi_fn : callable
#         Function with signature:
#             phi_hat, budget = find_phi_fn(phi, phi_max, **params)

#     param_grid : dict
#         Dictionary mapping parameter names -> list of values

#     R : int
#         Number of Monte Carlo trials per configuration

#     eps : float
#         Target absolute error threshold

#     phi_min, phi_max : float
#         Range for sampling phi

#     success_threshold : float
#         Required success rate (e.g. 0.95)

#     lambda_budget : float
#         Budget regularization weight for loss

#     rng : np.random.Generator
#         Random number generator

#     verbose : bool
#         Whether to print summary results

#     Returns
#     -------
#     grid_results : list of dict
#         Full results for all configurations
#     """

#     if rng is None:
#         rng = np.random.default_rng()

#     if phi_min is None:
#         phi_min = phi_max / 10

#     # Expand parameter grid
#     param_names = list(param_grid.keys())
#     param_values = list(param_grid.values())
#     param_combinations = list(product(*param_values))

#     grid_results = []
#     pbar = tqdm(param_combinations, desc="Grid search")

#     for values in pbar:
#         params = dict(zip(param_names, values))

#         errs = []
#         budgets = []
#         successes = []

#         for _ in range(R):
#             phi = rng.uniform(phi_min, phi_max)

#             phi_hat, budget = find_phi_fn(
#                 rng,
#                 phi,
#                 phi_max,
#                 phi_min,
#                 **params,
#             )

#             err = abs(phi_hat - phi)
#             errs.append(err)
#             budgets.append(budget)
#             successes.append(err <= eps)

#         mean_err = float(np.mean(errs))
#         mean_budget = float(np.mean(budgets))
#         success_rate = float(np.mean(successes))
#         loss = mean_err + lambda_budget * mean_budget

#         result = {
#             **params,
#             "mean_budget": mean_budget,
#             "err": mean_err,
#             "loss": loss,
#             "success_rate": success_rate,
#         }
#         grid_results.append(result)

#         pbar.set_postfix(
#             loss=f"{loss:.2e}",
#             success=f"{success_rate:.2f}",
#         )

#     # ----- Reporting -----
#     if verbose:
#         print("\n=== Grid search configuration ===")
#         print(f"R (trials per config): {R}")
#         print(f"eps: {eps:.1e}")
#         print(f"phi range: [{phi_min:.3e}, {phi_max:.3e}]")
#         print(f"success threshold: {success_threshold:.2f}")
#         print(f"lambda_budget: {lambda_budget:.1e}")
#         print("\nParameter grid:")
#         for k, v in param_grid.items():
#             print(f"  {k}: {v}")
#         print()
        
#         best_loss = min(grid_results, key=lambda r: r["loss"])

#         candidates = [
#             r for r in grid_results
#             if r["success_rate"] >= success_threshold
#         ]

#         print("\n=== Grid search summary ===")
#         print(f"Total configurations: {len(grid_results)}")
#         print(f"Candidates with success ≥ {success_threshold:.2f}: {len(candidates)}")

#         print("\nBest by loss:")
#         print(best_loss)

#         if candidates:
#             best_budget = min(candidates, key=lambda r: r["mean_budget"])
#             print("\nBest by budget (success-constrained):")
#             print(best_budget)
#         else:
#             print("\nNo configuration met the success threshold.")

#     return grid_results

# def evaluate_algorithm(
#     find_phi_fn,
#     *,
#     params,
#     R=1000,
#     phi_min=None,
#     phi_max=0.1,
#     eps=None,
#     rng=None,
# ):
#     """
#     Monte Carlo evaluation of a fixed algorithm configuration.

#     Parameters
#     ----------
#     find_phi_fn : callable
#         Function with signature:
#             phi_hat, budget = find_phi_fn(phi, phi_max, **params)

#     params : dict
#         Algorithm parameters

#     R : int
#         Number of independent simulations

#     phi_min, phi_max : float
#         Range for sampling phi

#     eps : float or None
#         If provided, compute success indicator |phi_hat - phi| <= eps

#     rng : np.random.Generator
#         Random number generator

#     Returns
#     -------
#     results : list of dict
#         Per-run results
#     """

#     if rng is None:
#         rng = np.random.default_rng()

#     if phi_min is None:
#         phi_min = phi_max / 10

#     results = []

#     for _ in range(R):
#         phi = rng.uniform(phi_min, phi_max)

#         phi_hat, budget_used = find_phi_fn(
#             rng,
#             phi,
#             phi_max,
#             phi_min,
#             **params,
#         )

#         result = {
#             "phi": phi,
#             "phi_hat": phi_hat,
#             "err": abs(phi_hat - phi),
#             "budget": budget_used,
#         }

#         if eps is not None:
#             result["success"] = result["err"] <= eps

#         results.append(result)


#     errs = np.array([r["err"] for r in results])
#     budgets = np.array([r["budget"] for r in results])

#     success_rate = np.mean(errs < eps)
#     mean_budget = np.mean(budgets)

#     print("Evaluation summary")
#     print("-" * 60)
#     print(f"Success criterion      : |φ̂ − φ| ≤ {eps:.1e}")
#     print(f"Success rate           : {success_rate:.3f} ({success_rate*100:.1f}%)")
#     print(f"Mean absolute error    : {errs.mean():.3e}")
#     print(f"Median absolute error  : {np.median(errs):.3e}")
#     print(f"Mean budget            : {mean_budget:,.0f}")
#     print(f"Median budget          : {np.median(budgets):,.0f}")
#     print("-" * 60)

#     return results

from grid_search import evaluate_algorithm

In [12]:
R = 10_000

params = {
    "m_exploration": 10_000,
    "m_exploitation": 10_000,
    "safeguard": 1,
    "conf": 0.95
}

results = evaluate_algorithm(
    find_phi_fn=find_phi_binary_search,
    params=params,
    R=R,
    phi_max=phi_max,
    eps=1e-3,
    rng=rng,
)


Using 4 parallel workers


Evaluating: 100%|██████████| 10000/10000 [00:05<00:00, 1996.46it/s]


Total runtime          : 0.09 minutes
Evaluation summary
------------------------------------------------------------
Success criterion      : |φ̂ − φ| ≤ 1.0e-03
Success rate           : 0.999 (99.9%)
Mean absolute error    : 1.576e-04
Median absolute error  : 1.084e-04
Mean budget            : 4,035,096
Median budget          : 3,160,000
------------------------------------------------------------


In [13]:
R = 10_000

params = {
    "m_exploration": 10_000,
    "m_exploitation": 10_000,
    "safeguard": 1,
    "conf": 0.95
}

results = evaluate_algorithm(
    find_phi_fn=find_phi_binary_search,
    params=params,
    R=R,
    phi_max=phi_max,
    eps=1e-3,
    rng=rng,
)


Using 4 parallel workers


Evaluating: 100%|██████████| 10000/10000 [00:04<00:00, 2307.53it/s]


Total runtime          : 0.08 minutes
Evaluation summary
------------------------------------------------------------
Success criterion      : |φ̂ − φ| ≤ 1.0e-03
Success rate           : 0.998 (99.8%)
Mean absolute error    : 1.617e-04
Median absolute error  : 1.106e-04
Mean budget            : 4,042,986
Median budget          : 3,160,000
------------------------------------------------------------


loss for err + budget

In [14]:
from grid_search import grid_search_algorithm

In [10]:
lambda_budget = 1e-7

# param_grid = {
#     "m_exploration": np.unique(np.logspace(2, 3, 8, dtype=int)),
#     "m_exploitation": np.unique(np.logspace(2, 4, 8, dtype=int)),
#     "conf": [0.9, 0.95, 0.975],
#     "safeguard": [0, 1, 2, 3],
# }

param_grid = {
    "m_exploration": [500, 1000, 2000],
    "m_exploitation": [10_000, 50_000, 100_000, 500_000],
    "conf": [0.95, 0.975],
    "safeguard": [0, 1, 2],
}

grid_results = grid_search_algorithm(
    find_phi_fn=find_phi_binary_search,
    param_grid=param_grid,
    R=1000,
    eps=1e-3,
    phi_max=0.1,
    phi_min=0.01,
    lambda_budget=1e-7,
    rng=rng,
)


Grid search: 100%|██████████| 72/72 [01:25<00:00,  1.18s/it, loss=1.96e+00, success=1.00]


=== Grid search configuration ===
R (trials per config): 1000
eps: 1.0e-03
phi range: [1.000e-02, 1.000e-01]
success threshold: 0.95
lambda_budget: 1.0e-07

Parameter grid:
  m_exploration: [500, 1000, 2000]
  m_exploitation: [10000, 50000, 100000, 500000]
  conf: [0.95, 0.975]
  safeguard: [0, 1, 2]


=== Grid search summary ===
Total configurations: 72
Candidates with success ≥ 0.95: 33

Best by loss:
{'m_exploration': 500, 'm_exploitation': 10000, 'conf': 0.95, 'safeguard': 2, 'mean_budget': 557670.5, 'err': 0.00017863037651802278, 'loss': 0.05594568037651802, 'success_rate': 0.992}

Best by budget (success-constrained):
{'m_exploration': 500, 'm_exploitation': 10000, 'conf': 0.95, 'safeguard': 2, 'mean_budget': 557670.5, 'err': 0.00017863037651802278, 'loss': 0.05594568037651802, 'success_rate': 0.992}


## Higher precision

In [ ]:
R = 300

params = {
    "m_exploration": 1000,
    "m_exploitation": 500_000,
    "safeguard": 2,
    "conf": 0.975
}

results = evaluate_algorithm(
    find_phi_fn=find_phi_binary_search,
    params=params,
    R=R,
    phi_max=phi_max,
    eps=1e-4,
    rng=rng,
)



In [ ]:
lambda_budget = 1e-7

# m_vals = [200, 400, 500, 1000, 2000]
m_vals = np.unique(
    np.logspace(2, 4, 25, dtype=int)
)
m_exploitation_vals = np.unique(
    np.logspace(4, 7, 25, dtype=int)
)
conf_vals = [0.9, 0.95, 0.975]
safeguard_vals = [0, 1, 2, 3]


param_grid = {
    "m_exploration": m_vals,
    "m_exploitation": m_exploitation_vals,
    "conf": conf_vals,
    "safeguard": safeguard_vals,
}


grid_results = grid_search_algorithm(
    find_phi_fn=find_phi_binary_search,
    param_grid=param_grid,
    R=300,
    eps=1e-3,
    phi_max=phi_max,
    lambda_budget=1e-7,
    rng=rng,
)



In [ ]:
max(grid_results, key=lambda x: x["success_rate"])

In [ ]:
candidates = [
    r for r in grid_results
    if r["success_rate"] >= 0.95
]

print(len(candidates))


best = min(candidates, key=lambda x: x["mean_budget"])
best

In [ ]:
print("hi")

In [ ]:
param_grid = {
    "m_exploration": np.unique(np.logspace(2, 3, 25, dtype=int)),
    "m_exploitation": np.unique(np.logspace(3, 4, 25, dtype=int)),
    "conf": [0.75, 0.8, 0.85],
    "safeguard": [1, 2, 5],
}

grid_results = grid_search_algorithm(
    find_phi_fn=find_phi_binary_search,
    param_grid=param_grid,
    R=1_000,
    eps=1e-3,
    phi_max=0.1,
    lambda_budget=1e-7,
    rng=rng,
)

In [ ]:
params = {
    "m_exploration": 127,
    "m_exploitation": 4000,
    "safeguard": 5,
    "conf": 0.8
}


results = evaluate_algorithm(
    find_phi_fn=find_phi_binary_search,
    params=params,
    R=10_000,
    phi_max=0.1,
    eps=1e-3,
    rng=rng,
)


# Smaller phi

In [ ]:
phi_max = 0.01

In [ ]:
R = 300

params = {
    "m_exploration": 100_000,
    "safeguard": 2,
    "conf": 0.95
}

results = []
for _ in range(R):
    phi = rng.uniform(phi_max/10, phi_max)
    N_opt = np.pi//(2*phi)

    N, budget_used = find_n_binary_search(phi, phi_max, **params)

    print(f"{phi = :.4f} {N_opt = }, {N = }")
    result = {
        "N_opt": N_opt,
        "N": N,
        "err": N_opt - N,
        "budget": budget_used
    }
    results.append(result)

errs = np.array([r["err"] for r in results])
budgets = np.array([r["budget"] for r in results])

plt.figure(figsize=(16, 8))

# symmetric bins around 0
# max_abs = max(abs(errs.min()), abs(errs.max()))
bins = np.arange(-errs.min() - 0.5, errs.max() + 1.5, 1)

plt.hist(errs, bins=bins, edgecolor="black", alpha=0.7)

# visual markers
plt.axvline(0, color="black", linestyle="--", linewidth=1)
plt.axvspan(bins[0], 0, color="red", alpha=0.15, label="Overshoot (N > N_opt)")
plt.axvspan(0, bins[-1], color="green", alpha=0.15, label="Undershoot (N < N_opt)")

plt.xlabel(r"$N_{\mathrm{opt}} - N$")
plt.ylabel("Frequency")
plt.title(
    rf"Binary search estimator error distribution ($R={R}$)"
)

plt.legend()
plt.tight_layout()
plt.show()

overshoot_rate = np.mean(errs < 0)
perfect_rate = np.mean(errs == 0)
mean_undershoot = np.mean(errs[errs > 0]) if np.any(errs > 0) else 0
mean_budget = np.mean(budgets)

print(f"Overshoot rate: {overshoot_rate:.2%}")
print(f"Exact hit rate: {perfect_rate:.2%}")
print(f"Mean undershoot: {mean_undershoot:.2}")
print(f"Mean budget: {mean_budget}")



Parameter grid search

In [ ]:
# lambda_budget = 1e-7

# m_vals = [200, 400, 500, 1000, 2000]
m_vals = np.unique(
    np.logspace(2, 2.5, 25, dtype=int)
)
m_exploitation_vals = np.unique(
    np.logspace(3, 4, 25, dtype=int)
)
conf_vals = [0.5, 0.65, 0.75]
safeguard_vals = [10, 15, 20]


# param_grid = list(product(
#     m_vals,
#     m_exploitation_vals,
#     conf_vals,
#     safeguard_vals
# ))

phi_max = 0.01

param_grid = {
    "m_exploration": m_vals,
    "m_exploitation": m_exploitation_vals,
    "conf": conf_vals,
    "safeguard": safeguard_vals,
}


grid_results = grid_search_algorithm(
    find_phi_fn=find_phi_binary_search,
    param_grid=param_grid,
    R=1000,
    eps=1e-4,
    phi_max=phi_max,
    lambda_budget=1e-7,
    rng=rng,
)

In [ ]:
params = {
    "m_exploration": 108,
    "m_exploitation": 3359,
    "safeguard": 10,
    "conf": 0.5
}


results = evaluate_algorithm(
    find_phi_fn=find_phi_binary_search,
    params=params,
    R=10_000,
    phi_max=0.01,
    eps=1e-4,
    rng=rng,
)


## Linear search

In [15]:
from linear_search import find_phi_linear_search

In [16]:
R = 500
phi_max = 0.1

params = {
    "m_exploration": 10_000,
    "m_exploitation": 10_000,
    "safeguard": 1,
    "lookback_window": 5,
}

results = evaluate_algorithm(
    find_phi_fn=find_phi_linear_search,
    params=params,
    R=R,
    phi_max=phi_max,
    eps=1e-3,
    rng=rng,
)

Using 4 parallel workers


Evaluating: 100%|██████████| 500/500 [00:00<00:00, 941.02it/s] 


Total runtime          : 0.01 minutes
Evaluation summary
------------------------------------------------------------
Success criterion      : |φ̂ − φ| ≤ 1.0e-03
Success rate           : 0.998 (99.8%)
Mean absolute error    : 1.818e-04
Median absolute error  : 1.339e-04
Mean budget            : 7,455,280
Median budget          : 3,230,000
------------------------------------------------------------


In [17]:
phi_max = 0.1

param_grid = {
    "m_exploration": np.unique(np.logspace(1, 3, 25, dtype=int)),
    "m_exploitation": np.unique(np.logspace(2, 4, 25, dtype=int)),
    "lookback_window": [1, 2, 5, 10],
    "safeguard": [0, 1, 2, 3],
}

grid_results = grid_search_algorithm(
    find_phi_fn=find_phi_linear_search,
    param_grid=param_grid,
    R=1000,
    eps=1e-3,
    phi_max=phi_max,
    lambda_budget=1e-7,
    rng=rng,
)


Using 4 parallel workers


Grid search | workers=4:   7%|▋         | 704/10000 [06:57<1:31:54,  1.69it/s, loss=1.34e-02, success=0.95]


KeyboardInterrupt: 

In [18]:
R = 10_000
phi_max = 0.1

params = {
    "m_exploration": 138,
    "m_exploitation": 3727,
    "safeguard": 0,
    "lookback_window": 1,
}

results = evaluate_algorithm(
    find_phi_fn=find_phi_linear_search,
    params=params,
    R=R,
    phi_max=phi_max,
    eps=1e-3,
    rng=rng,
)

Using 4 parallel workers


Evaluating: 100%|██████████| 10000/10000 [00:05<00:00, 1738.48it/s]


Total runtime          : 0.11 minutes
Evaluation summary
------------------------------------------------------------
Success criterion      : |φ̂ − φ| ≤ 1.0e-03
Success rate           : 0.938 (93.8%)
Mean absolute error    : 4.258e-04
Median absolute error  : 3.461e-04
Mean budget            : 66,436
Median budget          : 66,256
------------------------------------------------------------


Higher precision

In [ ]:
np.logspace(3, 6, 8, dtype=int)

In [ ]:
phi_max = 0.1

param_grid = {
    "m_exploration": np.unique(np.logspace(2, 4, 25, dtype=int)),
    "m_exploitation": np.unique(np.logspace(4, 6, 25, dtype=int)),
    "lookback_window": [1, 2],
    "safeguard": [1, 2],
}

grid_results = grid_search_algorithm(
    find_phi_fn=find_phi_linear_search,
    param_grid=param_grid,
    R=500,
    eps=1e-4,
    phi_max=phi_max,
    lambda_budget=1e-7,
    rng=rng,
)


In [19]:
params = {
    "m_exploration": 100,
    "m_exploitation": 517947,
    "safeguard": 2,
    "lookback_window": 1,
}

results = evaluate_algorithm(
    find_phi_fn=find_phi_linear_search,
    params=params,
    R=10_000,
    phi_max=0.1,
    eps=1e-4,
    rng=rng,
)

Using 4 parallel workers


Evaluating: 100%|██████████| 100000/100000 [03:54<00:00, 427.02it/s]


Total runtime          : 4.08 minutes
Evaluation summary
------------------------------------------------------------
Success criterion      : |φ̂ − φ| ≤ 1.0e-04
Success rate           : 0.953 (95.3%)
Mean absolute error    : 3.982e-05
Median absolute error  : 3.348e-05
Mean budget            : 7,264,215
Median budget          : 7,256,058
------------------------------------------------------------


smaller phi

In [ ]:
phi_max = 0.01

param_grid = {
    "m_exploration": np.unique(np.logspace(1.5, 2.5, 25, dtype=int)),
    "m_exploitation": np.unique(np.logspace(3, 4, 25, dtype=int)),
    "lookback_window": [1, 2],
    "safeguard": [25],
}

grid_results = grid_search_algorithm(
    find_phi_fn=find_phi_linear_search,
    param_grid=param_grid,
    R=1000,
    eps=1e-4,
    phi_max=phi_max,
    lambda_budget=1e-7,
    rng=rng,
)


In [20]:
params = {
    "m_exploration": 94,
    "m_exploitation": 5500,
    "safeguard": 25,
    "lookback_window": 1,
}

results = evaluate_algorithm(
    find_phi_fn=find_phi_linear_search,
    params=params,
    R=100_000,
    phi_max=0.01,
    eps=1e-4,
    rng=rng,
)

Using 4 parallel workers


Evaluating: 100%|██████████| 100000/100000 [01:29<00:00, 1121.58it/s]


Total runtime          : 1.72 minutes
Evaluation summary
------------------------------------------------------------
Success criterion      : |φ̂ − φ| ≤ 1.0e-04
Success rate           : 0.950 (95.0%)
Mean absolute error    : 4.062e-05
Median absolute error  : 3.433e-05
Mean budget            : 775,131
Median budget          : 755,610
------------------------------------------------------------


## Linear search with window

In [23]:
from linear_search import find_phi_linear_search_windowed

In [ ]:
phi_max = 0.1

param_grid = {
    "m_exploration": np.unique(np.logspace(1, 3, 25, dtype=int)),
    "m_exploitation": np.unique(np.logspace(2, 4, 25, dtype=int)),
    "window": [2, 3, 4, 5, 8],
    "lookback_window": [1, 2, 5, 10],
    "safeguard": [1, 2],
}

grid_results = grid_search_algorithm(
    find_phi_fn=find_phi_linear_search_windowed,
    param_grid=param_grid,
    R=1000,
    eps=1e-3,
    phi_max=phi_max,
    lambda_budget=1e-7,
    rng=rng,
    algorithm_name="linear_search_windowed",
)

Using 4 parallel workers


Grid search | workers=4:   3%|▎         | 784/25000 [09:37<5:30:12,  1.22it/s, loss=2.41e-02, success=0.79] 

In [ ]:

R = 10_000
phi_max = 0.1

params = {
    "m_exploration": 138,
    "m_exploitation": 3727,
    "safeguard": 0,
    "lookback_window": 1,
}

results = evaluate_algorithm(
    find_phi_fn=find_phi_linear_search_windowed,
    params=params,
    R=R,
    phi_max=phi_max,
    eps=1e-3,
    rng=rng,
)

Using 4 parallel workers


Evaluating: 100%|██████████| 10000/10000 [00:02<00:00, 4287.19it/s]

Total runtime          : 0.05 minutes
Evaluation summary
------------------------------------------------------------
Success criterion      : |φ̂ − φ| ≤ 1.0e-03
Success rate           : 0.935 (93.5%)
Mean absolute error    : 4.467e-04
Median absolute error  : 3.513e-04
Mean budget            : 66,433
Median budget          : 66,256
------------------------------------------------------------


## Reverse Engineering

In [7]:
from reverse_engineering import find_phi_reverse_engineering

In [ ]:
params = {
    "m_exploration": 10_000,
    "m_exploitation": 10_000,
    "safeguard": 0.9,
}

results = evaluate_algorithm(
    find_phi_fn=find_phi_reverse_engineering,
    params=params,
    R=10_000,
    phi_max=0.1,
    eps=1e-3,
    rng=rng,
)

In [ ]:
param_grid = {
    "m_exploration": np.unique(np.logspace(2, 3, 25, dtype=int)),
    "m_exploitation": np.unique(np.logspace(3, 5, 25, dtype=int)),
    "safeguard": [0.8, 0.9, 0.95, 1],
}

grid_results = grid_search_algorithm(
    find_phi_fn=find_phi_reverse_engineering,
    param_grid=param_grid,
    R=1000,
    eps=1e-3,
    phi_max=0.1,
    lambda_budget=1e-7,
    rng=rng,
)


Higher precision

In [ ]:
param_grid = {
    "m_exploration": np.unique(np.logspace(2.2, 3, 25, dtype=int)),
    "m_exploitation": np.unique(np.logspace(4, 6, 25, dtype=int)),
    "safeguard": [0.85, 0.9, 0.95],
}

grid_results = grid_search_algorithm(
    find_phi_fn=find_phi_reverse_engineering,
    param_grid=param_grid,
    R=1000,
    eps=1e-4,
    phi_max=0.1,
    lambda_budget=1e-7,
    rng=rng,
)


In [ ]:
params = {
    "m_exploration": 695,
    "m_exploitation": 183298,
    "safeguard": 0.9,
}

results = evaluate_algorithm(
    find_phi_fn=find_phi_reverse_engineering,
    params=params,
    R=10_000,
    phi_max=0.1,
    eps=1e-4,
    rng=rng,
)

Smaller phi

In [ ]:
param_grid = {
    "m_exploration": np.unique(np.logspace(2, 3, 25, dtype=int)),
    "m_exploitation": np.unique(np.logspace(3, 3.5, 25, dtype=int)),
    "safeguard": [0.85, 0.9, 0.95],
}

grid_results = grid_search_algorithm(
    find_phi_fn=find_phi_reverse_engineering,
    param_grid=param_grid,
    R=1000,
    eps=1e-4,
    phi_max=0.01,
    lambda_budget=1e-7,
    rng=rng,
)


In [ ]:
params = {
    "m_exploration": 545,
    "m_exploitation": 2500,
    "safeguard": 0.9,
}

results = evaluate_algorithm(
    find_phi_fn=find_phi_reverse_engineering,
    params=params,
    R=100_000,
    phi_max=0.01,
    eps=1e-4,
    rng=rng,
)

In [ ]:
params = {
    "m_exploration": 700,
    "m_exploitation": 1000,
    "safeguard": 0.9,
}

results = evaluate_algorithm(
    find_phi_fn=find_phi_reverse_engineering,
    params=params,
    R=100_000,
    phi_max=0.1,
    eps=1e-3,
    rng=rng,
)

## Brute Force

In [4]:
from brute_force import find_phi_brute_force

In [ ]:
R = 1000
phi_max = 0.1

params = {
    "m": 4_500,
}

results = evaluate_algorithm(
    find_phi_fn=find_phi_brute_force,
    params=params,
    R=10_000,
    phi_max=0.1,
    eps=1e-3,
    rng=rng,
)

Higher precision

In [ ]:
params = {
    "m": 430_000,
}

results = evaluate_algorithm(
    find_phi_fn=find_phi_brute_force,
    params=params,
    R=10_000,
    phi_max=0.1,
    eps=1e-4,
    rng=rng,
)

Smaller phi

In [ ]:
params = {
    "m": 4_000,
}

results = evaluate_algorithm(
    find_phi_fn=find_phi_brute_force,
    params=params,
    R=100_000,
    phi_max=0.01,
    eps=1e-4,
    rng=rng,
)

# Weirdly distributed phi

In [ ]:
param_grid = {
    "m_exploration": np.unique(np.logspace(2, 3, 25, dtype=int)),
    "m_exploitation": np.unique(np.logspace(3, 6, 25, dtype=int)),
    "safeguard": [0.85, 0.9, 0.95],
}

grid_results = grid_search_algorithm(
    find_phi_fn=find_phi_reverse_engineering,
    param_grid=param_grid,
    R=100,
    eps=1e-4,
    phi_max=0.1,
    phi_min=0.001,
    lambda_budget=1e-7,
    rng=rng,
)


In [ ]:
param_grid = {
    "m": np.unique(np.logspace(3, 6, 25, dtype=int)),
}

grid_results = grid_search_algorithm(
    find_phi_fn=find_phi_brute_force,
    param_grid=param_grid,
    R=100,
    eps=1e-4,
    phi_max=0.1,
    phi_min=0.001,
    lambda_budget=1e-7,
    rng=rng,
)


In [ ]:
param_grid = {
    "m_exploration": np.unique(np.logspace(2, 3, 25, dtype=int)),
    "m_exploitation": np.unique(np.logspace(3, 6, 25, dtype=int)),
    "lookback_window": [1, 2, 5],
    "safeguard": [0.85, 0.9, 0.95],
}

grid_results = grid_search_algorithm(
    find_phi_fn=find_phi_linear_search,
    param_grid=param_grid,
    R=100,
    eps=1e-4,
    phi_max=0.1,
    phi_min=0.001,
    lambda_budget=1e-7,
    rng=rng,
)


In [ ]:
param_grid = {
    "m_exploration": np.unique(np.logspace(2, 3, 25, dtype=int)),
    "m_exploitation": np.unique(np.logspace(3, 6, 25, dtype=int)),
    "conf": [0.5, 0.65, 0.75, 0.9],
    "safeguard": [5, 10, 15, 20],
}

grid_results = grid_search_algorithm(
    find_phi_fn=find_phi_binary_search,
    param_grid=param_grid,
    R=100,
    eps=1e-4,
    phi_max=0.1,
    phi_min=0.001,
    lambda_budget=1e-7,
    rng=rng,
)


# New binary algo

In [ ]:
def find_n_binary_search2(phi, phi_max, m_exploration=400, m_test=100, safeguard=1, conf=0.95):
    N_min = np.pi//(2*phi_max)
    N_max = np.pi//(2*phi_max/10)
    N = N_min
    
    phi_hat = simulate_errors(rng, phi, m_exploration, N)
    budget_used = m_exploration * N
    
    phi_1 = norm.ppf(1 - conf, phi_hat, np.sqrt(1 / (4 * m_exploration * N**2)))
    
    lb = N_min
    ub = N_max
    N += (ub - N) // 2
    
    done = False
    while not done:
        phi_hat = simulate_errors(rng, phi, m_test, N)
        budget_used += m_exploration * N
        
        temp_N = N
        if phi_hat < phi_1:
            N -= (N - lb) // 2
            ub = temp_N
        else:
            N += (ub - N) // 2
            lb = temp_N
            
            # update phi_1
            # phi_1 = norm.ppf(1 - conf, phi_hat, np.sqrt(1 / (4 * m_exploration * N**2)))
        
        if temp_N == N or N < N_min or N > N_max:
            done = True
    
    N -= safeguard # to avoid overshooting
    
    return N, budget_used

In [ ]:
R = 1000

params = {
    "m_exploration": 10_000,
    "safeguard": 1,
    "conf": 0.90
}

results = []
for _ in range(R):
    phi = rng.uniform(phi_max/10, phi_max)
    N_opt = np.pi//(2*phi)

    N, budget_used = find_n_binary_search2(phi, phi_max, **params)

    # print(f"{phi = :.3f} {N_opt = }, {N = }")
    result = {
        "N_opt": N_opt,
        "N": N,
        "err": N_opt - N,
        "budget": budget_used
    }
    results.append(result)
    
errs = np.array([r["err"] for r in results])
budgets = np.array([r["budget"] for r in results])

plt.figure(figsize=(16, 8))

# symmetric bins around 0
# max_abs = max(abs(errs.min()), abs(errs.max()))
bins = np.arange(-errs.min() - 0.5, errs.max() + 1.5, 1)

plt.hist(errs, bins=bins, edgecolor="black", alpha=0.7)

# visual markers
plt.axvline(0, color="black", linestyle="--", linewidth=1)
plt.axvspan(bins[0], 0, color="red", alpha=0.15, label="Overshoot (N > N_opt)")
plt.axvspan(0, bins[-1], color="green", alpha=0.15, label="Undershoot (N < N_opt)")

plt.xlabel(r"$N_{\mathrm{opt}} - N$")
plt.ylabel("Frequency")
plt.title(
    rf"Binary search estimator error distribution ($R={R}$)"
)

plt.legend()
plt.tight_layout()
plt.show()

overshoot_rate = np.mean(errs < 0)
perfect_rate = np.mean(errs == 0)
mean_undershoot = np.mean(errs[errs > 0]) if np.any(errs > 0) else 0
mean_budget = np.mean(budgets)

print(f"Overshoot rate: {overshoot_rate:.2%}")
print(f"Exact hit rate: {perfect_rate:.2%}")
print(f"Mean undershoot: {mean_undershoot:.2}")
print(f"Mean budget: {mean_budget}")



In [ ]:
def find_phi_binary_search2(phi, phi_max, phi_min, m_exploration=400, m_test=100, m_exploitation=1_000, safeguard=1, conf=0.95):
    N_min = np.pi//(2*phi_max)
    N_max = np.pi//(2*phi_min)
    N = N_min
    
    phi_hat = simulate_errors(rng, phi, m_exploration, N)
    budget_used = m_exploration * N
    
    phi_1 = norm.ppf(1 - conf, phi_hat, np.sqrt(1 / (4 * m_exploration * N**2)))
    
    lb = N_min
    ub = N_max
    N += (ub - N) // 2
    
    done = False
    while not done:
        phi_hat = simulate_errors(rng, phi, m_test, N)
        budget_used += m_exploration * N
        
        temp_N = N
        if phi_hat < phi_1:
            N -= (N - lb) // 2
            ub = temp_N
        else:
            N += (ub - N) // 2
            lb = temp_N
            
            # update phi_1
            # phi_1 = norm.ppf(1 - conf, phi_hat, np.sqrt(1 / (4 * m_exploration * N**2)))
        
        if temp_N == N or N < N_min or N > N_max:
            done = True
    
    N -= safeguard # to avoid overshooting
    
    phi_hat = simulate_errors(rng, phi, m_exploitation, N)
    budget_used += m_exploitation * N
    
    return phi_hat, budget_used

In [ ]:
lambda_budget = 1e-7

param_grid = {
    "m_exploration": np.unique(np.logspace(2, 3, 25, dtype=int)),
    "m_test": np.unique(np.logspace(1, 3, 25, dtype=int)),
    "m_exploitation": np.unique(np.logspace(2, 4, 25, dtype=int)),
    "conf": [0.9, 0.95, 0.975],
    "safeguard": [0, 1, 2, 3],
}

grid_results = grid_search_algorithm(
    find_phi_fn=find_phi_binary_search2,
    param_grid=param_grid,
    R=100,
    eps=1e-3,
    phi_max=0.1,
    phi_min=0.01,
    lambda_budget=1e-7,
    rng=rng,
)


# Simulating high precision selected, algorithms

In [11]:
from grid_search import grid_search_algorithm, evaluate_algorithm

In [22]:
params = {
    # "phi_max": 0.1,
    # "phi_min": 0.01,
    "m_exploration": 600,
    "m_exploitation": 500000,
    "conf": 0.95,
    # "eps": eps
}

results = evaluate_algorithm(
    find_phi_fn=find_phi_binary_search,
    params=params,
    R=100,
    phi_max=0.1,
    eps=1e-4,
    rng=rng,
)

Using 4 parallel workers


Evaluating: 100%|██████████| 100/100 [00:01<00:00, 76.85it/s]


Total runtime          : 0.02 minutes
Evaluation summary
------------------------------------------------------------
Success criterion      : |φ̂ − φ| ≤ 1.0e-04
Success rate           : 0.740 (74.0%)
Mean absolute error    : 3.081e-04
Median absolute error  : 2.937e-05
Mean budget            : 18,266,262
Median budget          : 13,174,000
------------------------------------------------------------


In [31]:
params = {
    # "phi_max": 0.1,
    # "phi_min": 0.01,
    "m_exploration": 1000,
    "m_exploitation": 100000,
    # "eps": eps
}

results = evaluate_algorithm(
    find_phi_fn=find_phi_reverse_engineering,
    params=params,
    R=100,
    phi_max=0.1,
    eps=1e-4,
    rng=rng,
)

Using 4 parallel workers


Evaluating: 100%|██████████| 100/100 [00:00<00:00, 191.99it/s]

Total runtime          : 0.01 minutes
Evaluation summary
------------------------------------------------------------
Success criterion      : |φ̂ − φ| ≤ 1.0e-04
Success rate           : 0.880 (88.0%)
Mean absolute error    : 5.491e-05
Median absolute error  : 3.102e-05
Mean budget            : 3,906,000
Median budget          : 2,965,000
------------------------------------------------------------
